In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

df = pd.read_excel(
    r"C:\Users\Path Bhimani\Downloads\Solar Power Generation Data.xlsx",
    usecols=[0],
    names=['Power']
)

df['Interval'] = (df.index % 96) + 1

df['Sine_Shape'] = np.sin((df['Interval'] - 24) * (np.pi / 48))
df['Cos_Shape'] = np.cos((df['Interval'] - 24) * (np.pi / 48))

df['Sine_Shape'] = df['Sine_Shape'].clip(lower=0)
df['Cos_Shape'] = df['Cos_Shape'].clip(lower=0)

df['Parabolic_Shape'] = 1 - ((df['Interval'] - 48) / 24) ** 2
df['Parabolic_Shape'] = df['Parabolic_Shape'].clip(lower=0)

df['parabolic_sine'] = df['Parabolic_Shape'] * df['Sine_Shape']

df['Same_Time_Yesterday'] = df['Power'].shift(96)
df['Same_Time_2_Days'] = df['Power'].shift(192)

df['Lag_1'] = df['Power'].shift(1)
df['Lag1_x_Sine'] = df['Lag_1'] * df['Sine_Shape']
df['Yesterday_x_Sine'] = df['Same_Time_Yesterday'] * df['Sine_Shape']

df['Gaussian_Sharp'] = np.exp(-((df['Interval'] - 48) ** 2) / (2 * (8 ** 2)))
df['Gaussian_sine'] = df['Gaussian_Sharp'] * df['Sine_Shape']

df['Sine_Squared'] = (np.sin((df['Interval'] - 24) * (np.pi / 48))) ** 2
df['Sine_Squared'] = df['Sine_Squared'].clip(lower=0)

df['Exp_Rise'] = np.exp((df['Interval'] - 48) / 20)
df['Exp_Rise'] = df['Exp_Rise'] / df['Exp_Rise'].max()
df['Exp_Rise'] = df['Exp_Rise'].clip(upper=1)

df['exp_rise_x_sine'] = df['Exp_Rise'] * df['Sine_Shape']

df['Double_Peak'] = (
    np.exp(-((df['Interval'] - 40) ** 2) / 100) +
    np.exp(-((df['Interval'] - 56) ** 2) / 100)
)
df['Double_Peak'] = df['Double_Peak'] / df['Double_Peak'].max()
df['Double_Peak_sine'] = df['Double_Peak'] * df['Sine_Shape']

x = (df['Interval'] - 48) / 48
df['Poly5'] = (1 - x**5).clip(lower=0)

df = df.dropna()

test_mask = (df.index >= 27840) & (df.index <= 30815)
train_mask = (df.index < 27840) | (df.index > 30815)

train_df = df[train_mask]
test_df = df[test_mask]

y_train = train_df['Power']
y_test = test_df['Power']

features = [
    'Same_Time_Yesterday', 'Same_Time_2_Days',
    'Sine_Shape', 'Cos_Shape', 'Parabolic_Shape',
    'Lag1_x_Sine', 'Yesterday_x_Sine', 'Double_Peak',
    'Gaussian_Sharp', 'Sine_Squared','Exp_Rise',
    'Poly5'
]

def recursive_forecast(model, df, features, test_df):
    preds = []
    steps_per_day = 96
    test_start_loc = df.index.get_loc(test_df.index[0])

    for i in range(0, len(test_df), steps_per_day):
        chunk_size = min(steps_per_day, len(test_df) - i)
        start_idx = test_start_loc + i
        history = df['Power'].iloc[start_idx-200:start_idx].tolist()

        for step in range(chunk_size):
            current_index = start_idx + step
            row = df.iloc[current_index].copy()

            if 'Lag_1' in features:
                row['Lag_1'] = history[-1]
            if 'Lag_2' in features:
                row['Lag_2'] = history[-2]

            if 'Rolling_Mean_4' in features:
                row['Rolling_Mean_4'] = np.mean(history[-4:])

            X_input = pd.DataFrame([row[features]])

            pred = model.predict(X_input)[0]
            pred = max(0, pred)
            pred = min(pred, 1.2 * max(history[-96:]))

            preds.append(pred)
            history.append(pred)

    return np.array(preds)

X_train = train_df[features]

model = RandomForestRegressor(
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=7,
    max_features=4,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = recursive_forecast(model, df, features, test_df)
y_pred = np.round(y_pred, 1)

print(f"R2 Score: {r2_score(y_test, y_pred):}")

plt.figure(figsize=(16,6))
plt.plot(y_test.values, label='Actual')
plt.plot(y_pred, '--', label='Forecast')
plt.title('Solar Power Forecast (Random Forest)')
plt.xlabel('Time Steps', fontsize=14, fontweight='bold')
plt.ylabel('Power', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
